In [1]:
import os, re
import json
import asyncio
import platform
from pydantic import BaseModel, Field
from typing import List, Dict, Any

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from pydantic import BaseModel, Field

from crawler_agent import LuoguCrawlerAgent
from crawl4ai import AsyncWebCrawler, BrowserConfig
from constant import *

from zai import ZhipuAiClient

deepseek = ChatOpenAI(
    model="deepseek-chat", 
    base_url=DMX_BASE_URL, 
    api_key=DMX_API_KEY, # 确保 LangChain 能拿到 Key
    temperature=0.1
)

glm = ZhipuAiClient(
    base_url=ZHIPU_BASE_URL, 
    api_key=ZHIPU_API_KEY,
)

In [2]:
class ProblemAnalysis(BaseModel):
    """用于分析算法题目的数据结构"""
    detailed_solution: str = Field(..., description="一份详细的、步骤清晰的题解")
    sample_code: str = Field(..., description="一份格式良好、有注释的 C++ 参考代码")
    keywords: List[str] = Field(..., description="解决此问题所需的核心知识点列表, 例如 ['动态规划', '01背包']")

print(ProblemAnalysis.model_fields)

{'detailed_solution': FieldInfo(annotation=str, required=True, description='一份详细的、步骤清晰的题解'), 'sample_code': FieldInfo(annotation=str, required=True, description='一份格式良好、有注释的 C++ 参考代码'), 'keywords': FieldInfo(annotation=List[str], required=True, description="解决此问题所需的核心知识点列表, 例如 ['动态规划', '01背包']")}


In [3]:
class AnalysisAgent:
    """
    不使用 with_structured_output 的实现：
    - 让 LLM 以 JSON 形式输出（在 prompt 中严格要求）
    - 提取 JSON，交给 Pydantic (ProblemAnalysis) 验证并解析
    """
    def __init__(self):
        self.llm = glm  # 你原来的 glm ChatOpenAI 实例
        self.schema = ProblemAnalysis  # Pydantic model
        print("[AnalysisAgent] 初始化完成，使用 ZhipuAI (glm-4.6)，不使用 with_structured_output。")

    def run(self, problem_data: Dict[str, Any], solutions_data: List[Dict[str, Any]]) -> Dict[str, Any]:
        print(f"[AnalysisAgent] 收到数据，开始分析...")
        system_prompt = f"""
你是一个算法竞赛金牌教练。
请你严格按照以下 JSON Schema 格式返回你的分析报告：
{ProblemAnalysis.model_fields}
"""
        user_prompt = f"""
请综合分析以下题目信息和多份爬取来的原始题解，生成一份全新的、高质量的分析报告。

[题目完整信息]:
{json.dump(problem_data, indent=2, ensure_ascii=False)}

[爬取的多份原始题解参考]:
{json.dump(solutions_data, indent=2, ensure_ascii=False)}

请只返回 JSON，不要任何多余文本。若无法完全确定某字段，也请返回空字符串或空列表，但必须是合法 JSON。
"""
        response = glm.chat.completions.create(
            model="glm-4.6", 
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            thinking={"type": "disabled"},
            # response_format={"type": "json_object"}, 
            stream=True
        )

        full_content = ""
        for chunk in response:
            if not chunk.choices:
                continue
            
            delta = chunk.choices[0].delta
            
            # 处理增量内容
            if hasattr(delta, 'content') and delta.content:
                full_content += delta.content
                print(delta.content, end="", flush=True)
            
            # 检查是否完成
            if chunk.choices[0].finish_reason:
                print(f"\n\n完成原因: {chunk.choices[0].finish_reason}")
                if hasattr(chunk, 'usage') and chunk.usage:
                    print(f"令牌使用: 输入 {chunk.usage.prompt_tokens}, 输出 {chunk.usage.completion_tokens}")

        return full_content

In [5]:
# 假设 AnalysisAgent, BrowserConfig, AsyncWebCrawler, LuoguCrawlerAgent 
# 已经在这个单元格之前被正确定义或导入。
#
import asyncio 
import traceback # <--- 1. 导入这个模块

TEST_PROBLEM_ID = "P4137" # 洛谷 P4137 [模板]可持久化线段树 2

# 1. 初始化分析 Agent (大脑)
analysis_agent = AnalysisAgent()

# 2. [Step 1] 将所有异步操作封装到一个 async 函数中
async def run_crawler_and_analysis():
    print("--- [Main] 启动爬虫 Agent ---\n") # 加个换行，好看点
    brouser_config = BrowserConfig(headless=True, proxy=None)
    
    raw_crawler_data = {}
    try:
        async with AsyncWebCrawler(config=brouser_config) as crawler:
            crawler_agent = LuoguCrawlerAgent(crawler)
            # 调用你的爬虫 Agent
            raw_crawler_data = await crawler_agent.run(TEST_PROBLEM_ID, max_solutions=1)
            
        print("--- [Main] 爬虫 Agent 执行完毕 ---")
    except Exception as e:
        # --- 2. 修改这里的打印方式 ---
        print(f"[Main] 爬虫 Agent 运行时发生异常，详细信息如下:")
        traceback.print_exc() # <--- 这会打印完整的错误堆栈！
        # --- 修改结束 ---
        return None

    # 3. 检查爬虫数据
    # (后续代码不变)
    problem_data = raw_crawler_data.get("problem", {})
    solutions_data = raw_crawler_data.get("solutions", [])
    
    if "error" in problem_data or not problem_data:
        print(f"[Main] 爬虫失败，无法启动分析 Agent: {problem_data.get('error', '未知错误')}")
        return None 

    # 4. [Step 2] 执行分析 Agent (大脑)
    print("--- [Main] 启动分析 Agent ---")
    
    result = analysis_agent.run(problem_data, solutions_data)
    
    print(result)
    return result

# 5. [Step 3] 在 Notebook 单元格的顶层 'await' 这个函数
await run_crawler_and_analysis()

[AnalysisAgent] 初始化完成，使用 ZhipuAI (glm-4.6)，不使用 with_structured_output。
--- [Main] 启动爬虫 Agent ---

[Main] 爬虫 Agent 运行时发生异常，详细信息如下:


Traceback (most recent call last):
  File "C:\Users\31761\AppData\Local\Temp\ipykernel_20724\1768836810.py", line 19, in run_crawler_and_analysis
    async with AsyncWebCrawler(config=brouser_config) as crawler:
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\anaconda3\envs\LuoguAgent\Lib\site-packages\crawl4ai\async_webcrawler.py", line 194, in __aenter__
    return await self.start()
           ^^^^^^^^^^^^^^^^^^
  File "d:\anaconda3\envs\LuoguAgent\Lib\site-packages\crawl4ai\async_webcrawler.py", line 177, in start
    await self.crawler_strategy.__aenter__()
  File "d:\anaconda3\envs\LuoguAgent\Lib\site-packages\crawl4ai\async_crawler_strategy.py", line 119, in __aenter__
    await self.start()
  File "d:\anaconda3\envs\LuoguAgent\Lib\site-packages\crawl4ai\async_crawler_strategy.py", line 129, in start
    await self.browser_manager.start()
  File "d:\anaconda3\envs\LuoguAgent\Lib\site-packages\crawl4ai\browser_manager.py", line 656, in start
    self.playwright =